In [ ]:
! pip install langchain langchain-openai langchain-community langchain-text-splitters langgraph python-dotenv faiss-cpu pypdf
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_core.tools import tool
from langgraph.graph import StateGraph, START
from typing import Annotated, TypedDict
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage, BaseMessage
from langgraph.prebuilt import ToolNode, tools_condition

In [3]:


load_dotenv()

llm = ChatOpenAI(model='gpt-4o-mini')

loader = PyPDFLoader("intro-to-ml.pdf")
docs = loader.load()

len(docs)

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(docs)


embeddings = OpenAIEmbeddings(model='text-embedding-3-small')
vector_store = FAISS.from_documents(chunks, embeddings)


retriever = vector_store.as_retriever(search_type='similarity', search_kwargs={'k':4})


@tool
def rag_tool(query):

    """
    Retrieve relevant information from the pdf document.
    Use this tool when the user asks factual / conceptual questions
    that might be answered from the stored documents.
    """
    result = retriever.invoke(query)

    context = [doc.page_content for doc in result]
    metadata = [doc.metadata for doc in result]

    return {
        'query': query,
        'context': context,
        'metadata': metadata
    }

tools = [rag_tool]
llm_with_tools = llm.bind_tools(tools)


class ChatState(TypedDict):
    messages: Annotated[list[BaseMessage], add_messages]



def chat_node(state: ChatState):

    messages = state['messages']

    response = llm_with_tools.invoke(messages)

    return {'messages': [response]}

tool_node = ToolNode(tools)


graph = StateGraph(ChatState)

graph.add_node('chat_node', chat_node)
graph.add_node('tools', tool_node)

graph.add_edge(START, 'chat_node')
graph.add_conditional_edges('chat_node', tools_condition)
graph.add_edge('tools', 'chat_node')

chatbot = graph.compile()



result = chatbot.invoke(
    {
        "messages": [
            HumanMessage(
                content=(
                    "Using the pdf notes, explain how to find the ideal value of K in KNN"
                )
            )
        ]
    }
)

print(result['messages'][-1].content)







To find the ideal value of \( K \) in the k-Nearest Neighbors (k-NN) algorithm, you should consider the following strategies:

1. **Understanding \( K \)**: The \( K \) in k-NN represents the number of nearest neighbors used when making predictions. Instead of relying on a single closest neighbor, \( K \) allows you to account for multiple neighbors when determining the class label for a new data point.

2. **Cross-Validation**: A common approach to determine the ideal \( K \) is to use cross-validation. This involves splitting your dataset into training and testing subsets multiple times, training the k-NN model on the training data with varying values of \( K \), and then evaluating the model performance on the test data. The value of \( K \) that yields the best performance (e.g., highest accuracy) is often chosen as the ideal value.

3. **Odd vs. Even Values of \( K \)**: It is generally advisable to choose an odd number for \( K \) to avoid ties in majority voting, especially in b